# 04 - Data Enrichment and Master Dataset Construction

## Objective

This notebook enriches the cleaned Airbnb listings datasets by combining listing-level data with review history, calendar availability, and neighbourhood-level metrics.

The objective is to create a consolidated master dataset containing one row per listing, suitable for analytics, dimensional modeling, machine learning segmentation, and dashboard reporting.

## Input

- Cleaned London listings dataset
- Cleaned New York listings dataset
- London and New York review datasets
- London and New York calendar datasets

## Output

- `london_master`
- `ny_master`
- `master_dataset`
- `master_airbnb_dataset.csv`

## Dataset Grain

The final master dataset is maintained at **one row per Airbnb listing**.

Review and calendar records are aggregated before joining to prevent row duplication and preserve listing-level grain.

In [1]:
import pandas as pd
import numpy as np

## 1. Load Enrichment Data Sources

The calendar and review datasets are loaded separately for London and New York.

These files contain multiple rows per listing:

- The calendar dataset stores daily availability records.
- The reviews dataset stores individual guest review records.

Because the cleaned listings dataset contains one row per listing, the calendar and review data must be summarized to listing level before joining.

In [2]:
london_listings = pd.read_csv("../data/raw/london/listings.csv.gz")
ny_listings = pd.read_csv("../data/raw/new_york/listings.csv.gz")

london_calendar = pd.read_csv("../data/raw/london/calendar.csv.gz")
ny_calendar = pd.read_csv("../data/raw/new_york/calendar.csv.gz")

london_reviews = pd.read_csv("../data/raw/london/reviews.csv.gz")
ny_reviews = pd.read_csv("../data/raw/new_york/reviews.csv.gz")

ArrowMemoryError: realloc of size 536870912 failed

In [ ]:
print("London calendar")
print(london_calendar.head())

print("\nNY calendar")
print(ny_calendar.head())

print("\nLondon reviews")
print(london_reviews.head())

print("\nNY reviews")
print(ny_reviews.head())

London calendar
            listing_id        date available  price  adjusted_price  \
0  1196288722069341420  2025-09-15         f    NaN             NaN   
1  1196288722069341420  2025-09-16         f    NaN             NaN   
2  1196288722069341420  2025-09-17         f    NaN             NaN   
3  1196288722069341420  2025-09-18         f    NaN             NaN   
4  1196288722069341420  2025-09-19         f    NaN             NaN   

   minimum_nights  maximum_nights  
0               1            1125  
1               1            1125  
2               1            1125  
3               1            1125  
4               1            1125  

NY calendar
   listing_id        date available  minimum_nights  maximum_nights
0       62925  2026-04-14         f              30              90
1       62925  2026-04-15         f              30              90
2       62925  2026-04-16         f              30              90
3       62925  2026-04-17         f              30     

## 2. Calendar Data Quality Assessment

The calendar datasets are inspected before aggregation to understand their completeness and available fields.

The London calendar extract contains no usable daily price values. Therefore, calendar-level revenue calculations cannot be performed reliably from this source.

The `available` field is used to derive an availability rate and an occupancy/unavailability proxy. However, unavailable dates may represent confirmed bookings, host-blocked dates, or other listing restrictions. Therefore, this metric should not be interpreted as confirmed occupancy.

In [ ]:
print("London calendar missing price:")
print(london_calendar["price"].isnull().mean() * 100)

print("\nLondon calendar available values:")
print(
    london_calendar["available"]
    .value_counts(dropna=False)
)

London calendar missing price:
100.0

London calendar available values:
available
f    21313866
t    14044108
Name: count, dtype: int64


In [ ]:
london_calendar[
    london_calendar["price"].notnull()
][
    ["listing_id", "date", "available", "price"]
].head(10)

,listing_id,date,available,price


In [ ]:
print("London Calendar Rows:")
print(len(london_calendar))

print("\nUnique London Listings:")
print(london_calendar["listing_id"].nunique())

print("\nNY Calendar Rows:")
print(len(ny_calendar))

print("\nUnique NY Listings:")
print(ny_calendar["listing_id"].nunique())

London Calendar Rows:
35357974

Unique London Listings:
96871

NY Calendar Rows:
12788141

Unique NY Listings:
35036


## 3. Review Aggregation

Guest reviews are aggregated from review-level records to listing-level metrics.

For each listing, the following enrichment fields are created:

- `total_reviews` - total number of reviews received
- `first_review_date` - earliest recorded review
- `last_review_date` - most recent recorded review

This creates one review-summary record per listing, allowing the information to be safely joined to the listings dataset.

In [ ]:
london_review_summary = (
    london_reviews
    .groupby("listing_id")
    .agg(
        total_reviews=("id", "count"),
        first_review_date=("date", "min"),
        last_review_date=("date", "max")
    )
    .reset_index()
)

ny_review_summary = (
    ny_reviews
    .groupby("listing_id")
    .agg(
        total_reviews=("id", "count"),
        first_review_date=("date", "min"),
        last_review_date=("date", "max")
    )
    .reset_index()
)

print("London Review Summary")
print(london_review_summary.head())

print("\nLondon Shape")
print(london_review_summary.shape)

print("\nNY Review Summary")
print(ny_review_summary.head())

print("\nNY Shape")
print(ny_review_summary.shape)

London Review Summary
   listing_id  total_reviews first_review_date last_review_date
0       13913             55        2010-08-18       2025-08-21
1       15400             97        2009-12-21       2025-04-05
2       17402             56        2011-03-21       2024-02-19
3       24328             95        2010-11-15       2025-07-05
4       36274             15        2011-03-20       2025-09-06

London Shape
(72749, 4)

NY Review Summary
   listing_id  total_reviews first_review_date last_review_date
0        6848            197        2009-05-25       2025-11-17
1        6872              2        2022-06-05       2025-10-07
2        6990            251        2009-10-28       2026-01-11
3        7097            423        2010-01-16       2025-09-23
4        7801             13        2009-08-09       2026-03-29

NY Shape
(24542, 4)


In [ ]:
print(london_review_summary.shape)
print(ny_review_summary.shape)

(72749, 4)
(24542, 4)


### Review Aggregation Result

The review datasets were successfully reduced from individual guest-review records to listing-level summaries.

Listings with no review history will not appear in the review summary. During the later left join, those listings remain in the master dataset with null review-summary fields, preserving the full listings population.

## 4. Calendar Aggregation

Daily calendar records are aggregated to listing level to create availability metrics.

For each listing, the following fields are calculated:

- `total_days` - number of calendar dates available in the extract
- `available_days` - number of dates marked as available
- `unavailable_days` - number of dates marked as unavailable
- `availability_rate` - available days divided by total calendar days
- `occupancy_rate` - unavailable days divided by total calendar days

The `occupancy_rate` field is treated as an **occupancy/unavailability proxy**, not confirmed booking occupancy.

In [ ]:
london_calendar_summary = (
    london_calendar
    .groupby("listing_id")
    .agg(
        total_days=("date", "count"),
        available_days=("available", lambda x: (x == "t").sum()),
        unavailable_days=("available", lambda x: (x == "f").sum())
    )
    .reset_index()
)

london_calendar_summary["availability_rate"] = (
    london_calendar_summary["available_days"]
    / london_calendar_summary["total_days"]
)

london_calendar_summary["occupancy_rate"] = (
    london_calendar_summary["unavailable_days"]
    / london_calendar_summary["total_days"]
)

In [ ]:
ny_calendar_summary = (
    ny_calendar
    .groupby("listing_id")
    .agg(
        total_days=("date", "count"),
        available_days=("available", lambda x: (x == "t").sum()),
        unavailable_days=("available", lambda x: (x == "f").sum())
    )
    .reset_index()
)

ny_calendar_summary["availability_rate"] = (
    ny_calendar_summary["available_days"] / ny_calendar_summary["total_days"]
)

ny_calendar_summary["occupancy_rate"] = (
    ny_calendar_summary["unavailable_days"]
    / ny_calendar_summary["total_days"]
)

In [ ]:
print(london_calendar_summary.head())
print()

print(london_calendar_summary.shape)
print()

print(ny_calendar_summary.head())
print()

print(ny_calendar_summary.shape)

   listing_id  total_days  available_days  unavailable_days  \
0       13913         365             331                34   
1       15400         365             199               166   
2       17402         365              80               285   
3       24328         365             294                71   
4       36274         365             323                42   

   availability_rate  occupancy_rate  
0           0.906849        0.093151  
1           0.545205        0.454795  
2           0.219178        0.780822  
3           0.805479        0.194521  
4           0.884932        0.115068  

(96871, 6)

   listing_id  total_days  available_days  unavailable_days  \
0        6848         365             138               227   
1        6872         365              83               282   
2        6990         365             273                92   
3        7097         365              45               320   
4        7801         365             193               172

### Calendar Aggregation Result

The calendar data was successfully transformed from daily records into one summary record per listing.

This reduces the London calendar dataset from more than 35 million daily records and the New York calendar dataset from more than 12 million daily records into manageable listing-level enrichment tables.

## 5. Listing-Level Master Dataset Construction

The cleaned listings datasets are enriched using left joins with the review and calendar summaries.

A left join is used to preserve every listing from the cleaned source dataset, including listings with no review history or incomplete calendar-related information.

The join structure is:

`Cleaned Listings → Review Summary → Calendar Summary`

In [ ]:
london_clean = pd.read_csv("../data/processed/london_listings_clean.csv")
ny_clean = pd.read_csv("../data/processed/ny_listings_clean.csv")

In [ ]:
london_master = (
    london_clean.merge(
        london_review_summary,
        left_on="id",
        right_on="listing_id",
        how="left"
    )
    .merge(
        london_calendar_summary,
        left_on="id",
        right_on="listing_id",
        how="left"
    )
)

print(london_master.shape)

london_master.head()

(96871, 89)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,listing_id_x,total_reviews,first_review_date,last_review_date,listing_id_y,total_days,available_days,unavailable_days,availability_rate,occupancy_rate
0,13913,https://www.airbnb.com/rooms/13913,20250914034649,2025-09-16,city scrape,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,Finsbury Park is a friendly melting pot commun...,https://a0.muscache.com/pictures/miso/Hosting-...,54730,...,13913.0,55.0,2010-08-18,2025-08-21,13913,365,331,34,0.906849,0.093151
1,15400,https://www.airbnb.com/rooms/15400,20250914034649,2025-09-16,city scrape,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,It is Chelsea.,https://a0.muscache.com/pictures/428392/462d26...,60302,...,15400.0,97.0,2009-12-21,2025-04-05,15400,365,199,166,0.545205,0.454795
2,17402,https://www.airbnb.com/rooms/17402,20250914034649,2025-09-16,city scrape,Very Central Modern 3-Bed/2 Bath By Oxford St W1,"You'll have a great time in this beautiful, cl...","Fitzrovia is a very desirable trendy, arty and...",https://a0.muscache.com/pictures/39d5309d-fba7...,67564,...,17402.0,56.0,2011-03-21,2024-02-19,17402,365,80,285,0.219178,0.780822
3,24328,https://www.airbnb.com/rooms/24328,20250914034649,2025-09-18,previous scrape,Battersea live/work artist house,"Artist house by SW Battersea Park, bright high...","- Battersea is a quiet family area, easy acces...",https://a0.muscache.com/pictures/9194b40f-c627...,41759,...,24328.0,95.0,2010-11-15,2025-07-05,24328,365,294,71,0.805479,0.194521
4,36274,https://www.airbnb.com/rooms/36274,20250914034649,2025-09-15,city scrape,Bright 1 bedroom apt off brick lane in Shoreditch,*Update June '25- Pump Installed to improve wa...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,133271,...,36274.0,15.0,2011-03-20,2025-09-06,36274,365,323,42,0.884932,0.115068


In [ ]:
london_master = london_master.drop(
    columns=["listing_id_x", "listing_id_y"]
)

print(london_master.shape)

(96871, 87)


In [ ]:
print("Duplicate listing IDs:", london_master["id"].duplicated().sum())

print(
    london_master[
        ["id", "price_clean", "total_reviews", "availability_rate", "occupancy_rate"]
    ].head()
)

Duplicate listing IDs: 0
      id  price_clean  total_reviews  availability_rate  occupancy_rate
0  13913         70.0           55.0           0.906849        0.093151
1  15400        149.0           97.0           0.545205        0.454795
2  17402        411.0           56.0           0.219178        0.780822
3  24328          NaN           95.0           0.805479        0.194521
4  36274        210.0           15.0           0.884932        0.115068


### Join Validation Result

Duplicate listing IDs are checked after enrichment to confirm that the aggregation and join strategy preserved one row per listing.

No duplicate listing IDs indicate that the master datasets maintain the intended listing-level grain and did not experience row multiplication during the joins.

In [ ]:
ny_master = (
    ny_clean.merge(
        ny_review_summary,
        left_on="id",
        right_on="listing_id",
        how="left"
    )
    .merge(
        ny_calendar_summary,
        left_on="id",
        right_on="listing_id",
        how="left"
    )
)

ny_master = ny_master.drop(
    columns=["listing_id_x", "listing_id_y"]
)

print(ny_master.shape)

print("Duplicate listing IDs:", ny_master["id"].duplicated().sum())

print(
    ny_master[
        ["id", "price_clean", "total_reviews", "availability_rate", "occupancy_rate"]
    ].head()
)

(35036, 100)
Duplicate listing IDs: 0
     id  price_clean  total_reviews  availability_rate  occupancy_rate
0  6848       117.27          197.0           0.378082        0.621918
1  6872        66.87            2.0           0.227397        0.772603
2  6990        77.17          251.0           0.747945        0.252055
3  7097       202.47          423.0           0.123288        0.876712
4  7801       365.97           13.0           0.528767        0.471233


## 6. Derived Analytical Features

Additional features are created to support analysis and segmentation.

- `review_frequency` estimates reviews per year of host tenure.
- `price_per_bedroom` standardizes listing price relative to bedroom count.
- `city` identifies the source market for each listing.

These features are created carefully because missing data differs across cities. For example, New York does not have usable host registration dates, so `host_tenure_years` and `review_frequency` remain unavailable for New York listings.

In [ ]:
london_master["review_frequency"] = (
    london_master["total_reviews"]
    / london_master["host_tenure_years"]
)

ny_master["review_frequency"] = (
    ny_master["total_reviews"]
    / ny_master["host_tenure_years"]
)

In [ ]:
london_master["price_per_bedroom"] = (
    london_master["price_clean"]
    / london_master["beds"]
)

ny_master["price_per_bedroom"] = (
    ny_master["price_clean"]
    / ny_master["beds"]
)

In [ ]:
london_master["city"] = "London"
ny_master["city"] = "New York"

In [ ]:
print(
    london_master[
        [
            "price_clean",
            "beds",
            "price_per_bedroom",
            "total_reviews",
            "host_tenure_years",
            "review_frequency"
        ]
    ].head()
)

print()

print(
    ny_master[
        [
            "price_clean",
            "beds",
            "price_per_bedroom",
            "total_reviews",
            "host_tenure_years",
            "review_frequency"
        ]
    ].head()
)

   price_clean  beds  price_per_bedroom  total_reviews  host_tenure_years  \
0         70.0   1.0               70.0           55.0          15.843836   
1        149.0   1.0              149.0           97.0          15.791781   
2        411.0   3.0              137.0           56.0          15.709589   
3          NaN   NaN                NaN           95.0          15.983562   
4        210.0   0.0                inf           15.0          15.315068   

   review_frequency  
0          3.471382  
1          6.142436  
2          3.564702  
3          5.943606  
4          0.979428  

   price_clean  beds  price_per_bedroom  total_reviews  host_tenure_years  \
0       117.27   NaN                NaN          197.0                NaN   
1        66.87   NaN                NaN            2.0                NaN   
2        77.17   NaN                NaN          251.0                NaN   
3       202.47   NaN                NaN          423.0                NaN   
4       365.97   Na

In [ ]:
london_master.loc[
    london_master["beds"] <= 0,
    "price_per_bedroom"
] = np.nan

ny_master.loc[
    ny_master["beds"] <= 0,
    "price_per_bedroom"
] = np.nan

In [ ]:
print(
    "London infinite values:",
    np.isinf(london_master["price_per_bedroom"]).sum()
)

print(
    "NY infinite values:",
    np.isinf(ny_master["price_per_bedroom"]).sum()
)

London infinite values: 0
NY infinite values: 0


### Derived Feature Validation

Listings with zero or missing bedroom counts cannot produce a meaningful price-per-bedroom value.

Such records are retained in the master dataset, but `price_per_bedroom` is set to null where the calculation would be invalid. This avoids infinite or misleading values in later analytics and machine learning workflows.

## 7. Cross-City Dataset Harmonization

London and New York have a shared set of core analytical fields but also contain city-specific source attributes.

The datasets are combined using a union-based approach. Shared fields support cross-city analytics, while city-specific fields remain available where present.

Raw price values are retained in the currency representation provided by the source extracts. Therefore, raw average-price comparisons between cities should not be interpreted as currency-normalized financial comparisons.

In [ ]:
set(london_master.columns) - set(ny_master.columns)

set(ny_master.columns) - set(london_master.columns)

{'host_profile_id',
 'host_profile_url',
 'hosts_time_as_host_months',
 'hosts_time_as_host_years',
 'hosts_time_as_user_months',
 'hosts_time_as_user_years',
 'license',
 'neighbourhood_group_cleansed',
 'price_quote_checkin_date',
 'price_quote_checkout_date',
 'price_quote_price_per_night',
 'price_quote_raw',
 'price_quote_total_price'}

In [ ]:
set(london_master.columns) - set(ny_master.columns)

set()

In [ ]:
london_master["city"] = "London"
ny_master["city"] = "New York"

In [ ]:
master_dataset = pd.concat(
    [london_master, ny_master],
    ignore_index=True
)

print(master_dataset.shape)

(131907, 103)


In [ ]:
print(
    master_dataset["city"]
    .value_counts()
)

print()

print(
    master_dataset[
        [
            "city",
            "id",
            "price_clean",
            "total_reviews",
            "availability_rate"
        ]
    ].head(50)
)

city
London      96871
New York    35036
Name: count, dtype: int64

      city     id  price_clean  total_reviews  availability_rate
0   London  13913         70.0           55.0           0.906849
1   London  15400        149.0           97.0           0.545205
2   London  17402        411.0           56.0           0.219178
3   London  24328          NaN           95.0           0.805479
4   London  36274        210.0           15.0           0.884932
5   London  36299        280.0          116.0           0.887671
6   London  36660         90.0          730.0           0.791781
7   London  38605         61.0          387.0           0.024658
8   London  38610        340.0           42.0           0.868493
9   London  38995         49.0           72.0           0.471233
10  London  39387          NaN           10.0           0.000000
11  London  41445        213.0           25.0           0.443836
12  London  41509          NaN           71.0           0.000000
13  London  41712     

In [ ]:
master_dataset.shape

master_dataset["city"].value_counts()

city
London      96871
New York    35036
Name: count, dtype: int64

In [ ]:
print(master_dataset.shape)

(131907, 103)


## 8. Neighbourhood-Level Enrichment

Neighbourhood-level metrics are calculated from the master dataset and joined back to each listing.

The following neighbourhood context fields are created:

- `neighbourhood_listing_count`
- `neighbourhood_median_price`
- `neighbourhood_avg_rating`

This adds local-market context to each listing and supports neighbourhood-based analytics in later SQL queries and dashboard visualizations.

In [ ]:
neighbourhood_
 = (
    master_dataset
    .groupby(["city", "neighbourhood_cleansed"])
    .agg(
        neighbourhood_listing_count=("id", "count"),
        neighbourhood_median_price=("price_clean", "median"),
        neighbourhood_avg_rating=("review_scores_rating", "mean")
    )
    .reset_index()
)

print(neighbourhood_agg.head())
print(neighbourhood_agg.shape)

     city neighbourhood_cleansed  neighbourhood_listing_count  \
0  London   Barking and Dagenham                          750   
1  London                 Barnet                         2529   
2  London                 Bexley                          640   
3  London                  Brent                         3019   
4  London                Bromley                          913   

   neighbourhood_median_price  neighbourhood_avg_rating  
0                        82.0                  4.576369  
1                       107.0                  4.655158  
2                        70.0                  4.735688  
3                       107.0                  4.668933  
4                        88.0                  4.774824  
(257, 5)


In [ ]:
master_dataset = master_dataset.merge(
    neighbourhood_agg,
    on=["city", "neighbourhood_cleansed"],
    how="left"
)

print(master_dataset.shape)

(131907, 106)


In [ ]:
master_dataset.to_csv("../data/curated/master_airbnb_dataset.csv", index=False)

In [ ]:
print("Master dataset saved successfully")
print(master_dataset.shape)

Master dataset saved successfully
(131907, 106)


## Enrichment Summary and Next Steps

The enrichment process created a unified master dataset containing 131,907 Airbnb listings across London and New York.

Key outcomes:

1. Review-level records were aggregated into listing-level review metrics.
2. Daily calendar records were aggregated into availability and occupancy/unavailability proxy metrics.
3. Cleaned listings were enriched through validated left joins.
4. The one-row-per-listing grain was preserved after enrichment.
5. Derived features were created for price-per-bedroom, review frequency, and city identification.
6. Neighbourhood-level metrics were added to provide local market context.
7. The final master dataset was saved to the curated data layer.

The next notebook uses the master dataset to create a dimensional model for analytical querying.